In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone

BRONZE_SCHEMA = "supplysage_bronze"
SILVER_SCHEMA = "supplysage_silver"

SOURCE_TABLE = f"{BRONZE_SCHEMA}.bronze_dataco_supply_chain"
TARGET_TABLE = f"{SILVER_SCHEMA}.silver_dataco_orders_shipments"

TRANSFORM_RUN_ID = f"silver_dataco_orders_shipments_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "11_silver_dataco_orders_shipments"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

print("TRANSFORM_RUN_ID:", TRANSFORM_RUN_ID)
print("SOURCE_TABLE:", SOURCE_TABLE)
print("TARGET_TABLE:", TARGET_TABLE)

bronze_dataco_df = spark.table(SOURCE_TABLE)

print("Bronze DataCo row count:", bronze_dataco_df.count())

display(bronze_dataco_df.limit(10))
bronze_dataco_df.printSchema()

In [0]:
# Notebook 11, Chunk 2: Inspect DataCo columns and create safe resolver

dataco_columns = bronze_dataco_df.columns

print("Total DataCo columns:", len(dataco_columns))

for idx, col_name in enumerate(dataco_columns, start=1):
    print(f"{idx}. {col_name}")


def resolve_col(possible_names):
    """
    Finds the actual source column name from a list of possible column names.
    Handles minor case/spacing differences.
    """
    normalized_map = {
        col_name.lower().strip(): col_name
        for col_name in bronze_dataco_df.columns
    }

    for name in possible_names:
        key = name.lower().strip()
        if key in normalized_map:
            return normalized_map[key]

    raise ValueError(f"None of these columns were found: {possible_names}")


# Quick checks for the columns we expect to use in Silver.
required_dataco_source_columns = {
    "order_id": ["Order Id", "order_id"],
    "order_item_id": ["Order Item Id", "order_item_id"],
    "order_date": ["order date (DateOrders)", "Order Date", "order_date"],
    "shipping_date": ["shipping date (DateOrders)", "Shipping Date", "shipping_date"],
    "delivery_status": ["Delivery Status", "delivery_status"],
    "late_delivery_risk": ["Late_delivery_risk", "late_delivery_risk"],
    "days_shipping_real": ["Days for shipping (real)", "days_for_shipping_real"],
    "days_shipping_scheduled": ["Days for shipment (scheduled)", "days_for_shipment_scheduled"],
    "shipping_mode": ["Shipping Mode", "shipping_mode"],
    "order_status": ["Order Status", "order_status"],
    "market": ["Market", "market"],
    "order_region": ["Order Region", "order_region"],
    "order_country": ["Order Country", "order_country"],
    "order_state": ["Order State", "order_state"],
    "order_city": ["Order City", "order_city"],
    "customer_segment": ["Customer Segment", "customer_segment"],
    "category_name": ["Category Name", "category_name"],
    "department_name": ["Department Name", "department_name"],
    "product_name": ["Product Name", "product_name"],
    "product_card_id": ["Product Card Id", "Product Card ID", "product_card_id"],
    "order_item_quantity": ["Order Item Quantity", "order_item_quantity"],
    "sales": ["Sales", "sales"],
    "order_item_total": ["Order Item Total", "order_item_total"],
    "order_profit_per_order": ["Order Profit Per Order", "order_profit_per_order"],
    "benefit_per_order": ["Benefit per order", "benefit_per_order"],
    "sales_per_customer": ["Sales per customer", "sales_per_customer"]
}

resolved_dataco_columns = {}

for target_name, possible_names in required_dataco_source_columns.items():
    resolved_dataco_columns[target_name] = resolve_col(possible_names)

print("Resolved DataCo source columns:")
for target_name, source_col_name in resolved_dataco_columns.items():
    print(f"{target_name} -> {source_col_name}")

In [0]:
def clean_string_expr(column_expr):
    return (
        F.when(column_expr.isNull(), None)
        .when(F.trim(column_expr) == "", None)
        .when(F.lower(F.trim(column_expr)).isin("null", "none", "nan"), None)
        .otherwise(F.trim(column_expr))
    )


def src_col(target_name):
    return F.col(f"`{resolved_dataco_columns[target_name]}`")


def parse_dataco_timestamp(column_expr):
    return F.coalesce(
        F.to_timestamp(column_expr, "M/d/yyyy H:mm"),
        F.to_timestamp(column_expr, "M/d/yyyy HH:mm"),
        F.to_timestamp(column_expr, "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(column_expr)
    )


silver_dataco_df = (
    bronze_dataco_df
    .select(
        src_col("order_id").cast("long").alias("order_id"),
        src_col("order_item_id").cast("long").alias("order_item_id"),

        parse_dataco_timestamp(src_col("order_date")).alias("order_timestamp"),
        parse_dataco_timestamp(src_col("shipping_date")).alias("shipping_timestamp"),

        clean_string_expr(src_col("delivery_status")).alias("delivery_status"),
        src_col("late_delivery_risk").cast("int").alias("late_delivery_risk_flag"),
        src_col("days_shipping_real").cast("int").alias("days_for_shipping_real"),
        src_col("days_shipping_scheduled").cast("int").alias("days_for_shipment_scheduled"),
        clean_string_expr(src_col("shipping_mode")).alias("shipping_mode"),
        clean_string_expr(src_col("order_status")).alias("order_status"),

        clean_string_expr(src_col("market")).alias("market"),
        clean_string_expr(src_col("order_region")).alias("order_region"),
        clean_string_expr(src_col("order_country")).alias("order_country"),
        clean_string_expr(src_col("order_state")).alias("order_state"),
        clean_string_expr(src_col("order_city")).alias("order_city"),

        clean_string_expr(src_col("customer_segment")).alias("customer_segment"),

        clean_string_expr(src_col("department_name")).alias("department_name"),
        clean_string_expr(src_col("category_name")).alias("category_name"),
        src_col("product_card_id").cast("long").alias("dataco_product_card_id"),
        clean_string_expr(src_col("product_name")).alias("product_name"),

        src_col("order_item_quantity").cast("int").alias("order_item_quantity"),
        src_col("sales").cast("double").alias("sales"),
        src_col("order_item_total").cast("double").alias("order_item_total"),
        src_col("order_profit_per_order").cast("double").alias("order_profit_per_order"),
        src_col("benefit_per_order").cast("double").alias("benefit_per_order"),
        src_col("sales_per_customer").cast("double").alias("sales_per_customer")
    )
    .withColumn("order_date", F.to_date(F.col("order_timestamp")))
    .withColumn("shipping_date", F.to_date(F.col("shipping_timestamp")))
    .withColumn("order_date_id", F.date_format(F.col("order_date"), "yyyyMMdd").cast("int"))
    .withColumn("shipping_date_id", F.date_format(F.col("shipping_date"), "yyyyMMdd").cast("int"))
    .withColumn("order_year", F.year(F.col("order_date")))
    .withColumn("order_month", F.month(F.col("order_date")))
    .withColumn(
        "shipping_delay_days",
        F.col("days_for_shipping_real") - F.col("days_for_shipment_scheduled")
    )
    .withColumn(
        "is_late_delivery",
        F.when(F.col("late_delivery_risk_flag") == 1, F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn(
        "is_delayed_beyond_schedule",
        F.when(F.col("shipping_delay_days") > 0, F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn(
        "is_cancelled_or_suspect_order",
        F.when(
            F.lower(F.col("order_status")).isin("canceled", "suspected_fraud"),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "profit_margin",
        F.when(F.col("sales") != 0, F.col("order_profit_per_order") / F.col("sales"))
        .otherwise(None)
    )
    .withColumn(
        "order_shipment_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("order_id").cast("string"),
                F.col("order_item_id").cast("string"),
                F.col("dataco_product_card_id").cast("string")
            ),
            256
        )
    )
    .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("silver_created_by", F.lit(CREATED_BY))
    .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
)

silver_dataco_row_count = silver_dataco_df.count()

print("Silver DataCo orders/shipments row count:", silver_dataco_row_count)

display(
    silver_dataco_df
    .select(
        "order_shipment_record_id",
        "order_id",
        "order_item_id",
        "order_date",
        "shipping_date",
        "delivery_status",
        "late_delivery_risk_flag",
        "shipping_delay_days",
        "is_late_delivery",
        "shipping_mode",
        "order_status",
        "market",
        "order_region",
        "order_country",
        "category_name",
        "product_name",
        "order_item_quantity",
        "sales",
        "profit_margin"
    )
    .limit(20)
)

silver_dataco_df.printSchema()

In [0]:
dataco_validation_rows = []

source_row_count = bronze_dataco_df.count()
target_row_count = silver_dataco_df.count()

dataco_validation_rows.append({
    "validation_check": "row_count_preserved",
    "expected_value": str(source_row_count),
    "actual_value": str(target_row_count),
    "status": "PASS" if source_row_count == target_row_count else "FAIL",
    "issue_detail": None if source_row_count == target_row_count else "Silver DataCo row count does not match Bronze row count."
})

null_key_count = (
    silver_dataco_df
    .filter(
        F.col("order_id").isNull()
        | F.col("order_item_id").isNull()
        | F.col("dataco_product_card_id").isNull()
    )
    .count()
)

dataco_validation_rows.append({
    "validation_check": "business_keys_not_null",
    "expected_value": "0 null key rows",
    "actual_value": str(null_key_count),
    "status": "PASS" if null_key_count == 0 else "FAIL",
    "issue_detail": None if null_key_count == 0 else "Null values found in order_id, order_item_id, or dataco_product_card_id."
})

duplicate_grain_count = (
    silver_dataco_df
    .groupBy("order_id", "order_item_id", "dataco_product_card_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

dataco_validation_rows.append({
    "validation_check": "grain_unique_order_item_product",
    "expected_value": "0 duplicate grain rows",
    "actual_value": str(duplicate_grain_count),
    "status": "PASS" if duplicate_grain_count == 0 else "FAIL",
    "issue_detail": None if duplicate_grain_count == 0 else "Duplicate rows found for order_id + order_item_id + dataco_product_card_id."
})

invalid_late_risk_flag_count = (
    silver_dataco_df
    .filter(~F.col("late_delivery_risk_flag").isin(0, 1))
    .count()
)

dataco_validation_rows.append({
    "validation_check": "late_delivery_risk_flag_valid",
    "expected_value": "Only 0 or 1",
    "actual_value": str(invalid_late_risk_flag_count),
    "status": "PASS" if invalid_late_risk_flag_count == 0 else "FAIL",
    "issue_detail": None if invalid_late_risk_flag_count == 0 else "late_delivery_risk_flag contains values outside 0 or 1."
})

negative_shipping_days_count = (
    silver_dataco_df
    .filter(
        (F.col("days_for_shipping_real") < 0)
        | (F.col("days_for_shipment_scheduled") < 0)
    )
    .count()
)

dataco_validation_rows.append({
    "validation_check": "shipping_days_non_negative",
    "expected_value": "0 negative shipping day rows",
    "actual_value": str(negative_shipping_days_count),
    "status": "PASS" if negative_shipping_days_count == 0 else "FAIL",
    "issue_detail": None if negative_shipping_days_count == 0 else "Negative values found in shipping day fields."
})

invalid_quantity_count = (
    silver_dataco_df
    .filter(F.col("order_item_quantity") <= 0)
    .count()
)

dataco_validation_rows.append({
    "validation_check": "order_item_quantity_positive",
    "expected_value": "0 non-positive quantity rows",
    "actual_value": str(invalid_quantity_count),
    "status": "PASS" if invalid_quantity_count == 0 else "FAIL",
    "issue_detail": None if invalid_quantity_count == 0 else "order_item_quantity has zero or negative values."
})

null_order_date_count = (
    silver_dataco_df
    .filter(F.col("order_date").isNull())
    .count()
)

dataco_validation_rows.append({
    "validation_check": "order_date_not_null",
    "expected_value": "0 null order dates",
    "actual_value": str(null_order_date_count),
    "status": "PASS" if null_order_date_count == 0 else "FAIL",
    "issue_detail": None if null_order_date_count == 0 else "Null order_date values found."
})

null_record_id_count = (
    silver_dataco_df
    .filter(F.col("order_shipment_record_id").isNull())
    .count()
)

dataco_validation_rows.append({
    "validation_check": "order_shipment_record_id_not_null",
    "expected_value": "0 null record ids",
    "actual_value": str(null_record_id_count),
    "status": "PASS" if null_record_id_count == 0 else "FAIL",
    "issue_detail": None if null_record_id_count == 0 else "Null order_shipment_record_id values found."
})

pii_columns_found = [
    col_name for col_name in silver_dataco_df.columns
    if any(
        pii_token in col_name.lower()
        for pii_token in ["email", "fname", "lname", "password", "street", "zipcode", "latitude", "longitude"]
    )
]

dataco_validation_rows.append({
    "validation_check": "pii_columns_excluded_from_silver",
    "expected_value": "No direct customer PII columns",
    "actual_value": ", ".join(pii_columns_found) if pii_columns_found else "None",
    "status": "PASS" if len(pii_columns_found) == 0 else "FAIL",
    "issue_detail": None if len(pii_columns_found) == 0 else "PII-like columns found in Silver DataCo output."
})

dataco_validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

dataco_validation_df = spark.createDataFrame(
    dataco_validation_rows,
    schema=dataco_validation_schema
)

display(dataco_validation_df.orderBy("status", "validation_check"))

fail_count = dataco_validation_df.filter(F.col("status") == "FAIL").count()

print("Silver DataCo validation failures:", fail_count)

In [0]:
if fail_count == 0:
    (
        silver_dataco_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"✅ Wrote Silver DataCo orders/shipments table: {TARGET_TABLE}")
else:
    raise ValueError("Silver DataCo validation failed. Fix issues before writing.")

In [0]:
written_dataco_df = spark.table(TARGET_TABLE)

written_row_count = written_dataco_df.count()

print("Written Silver DataCo row count:", written_row_count)

display(
    written_dataco_df
    .select(
        "order_shipment_record_id",
        "order_id",
        "order_item_id",
        "order_date",
        "shipping_date",
        "delivery_status",
        "late_delivery_risk_flag",
        "days_for_shipping_real",
        "days_for_shipment_scheduled",
        "shipping_delay_days",
        "is_late_delivery",
        "is_delayed_beyond_schedule",
        "shipping_mode",
        "order_status",
        "market",
        "order_region",
        "order_country",
        "customer_segment",
        "category_name",
        "product_name",
        "order_item_quantity",
        "sales",
        "profit_margin"
    )
    .limit(20)
)

if written_row_count == 180519:
    print("✅ Read-back check passed: Silver DataCo orders/shipments table exists and row count is correct.")
else:
    print("❌ Read-back check failed: Row count does not match expected 180,519.")

In [0]:
dataco_validation_results_df = (
    dataco_validation_df
    .withColumn("transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("target_table", F.lit(TARGET_TABLE))
    .withColumn("validated_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("created_by", F.lit(CREATED_BY))
    .withColumn("notebook_name", F.lit(NOTEBOOK_NAME))
    .select(
        "transform_run_id",
        "source_table",
        "target_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "status",
        "issue_detail",
        "validated_at",
        "created_by",
        "notebook_name"
    )
)

validation_results_table = f"{SILVER_SCHEMA}.silver_transform_validation_results"

try:
    spark.sql(f"""
    DELETE FROM {validation_results_table}
    WHERE transform_run_id = '{TRANSFORM_RUN_ID}'
    """)
except Exception as e:
    print("Validation results table does not exist yet. It will be created now.")

(
    dataco_validation_results_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

display(
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .groupBy("status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("status")
)

print("✅ Notebook 11 PASSED: silver_dataco_orders_shipments created and validated.")
print("Next notebook: 12_silver_synthetic_internal_tables")

In [0]:
# Notebook 11, Chunk 8: Clean DataCo source field dictionary

DESCRIPTION_SOURCE_TABLE = f"{BRONZE_SCHEMA}.bronze_dataco_description"
DESCRIPTION_TARGET_TABLE = f"{SILVER_SCHEMA}.silver_dataco_field_dictionary"

bronze_description_df = spark.table(DESCRIPTION_SOURCE_TABLE)

print("Bronze DataCo description row count:", bronze_description_df.count())

display(bronze_description_df.limit(20))
bronze_description_df.printSchema()

In [0]:
# Notebook 11, Chunk 9: Create cleaned Silver DataCo field dictionary dataframe

def clean_string_col_from_description(col_name):
    return (
        F.when(F.col(col_name).isNull(), None)
        .when(F.trim(F.col(col_name)) == "", None)
        .when(F.lower(F.trim(F.col(col_name))).isin("null", "none", "nan"), None)
        .otherwise(F.trim(F.col(col_name)))
    )


silver_dataco_dictionary_df = (
    bronze_description_df
    .select(
        clean_string_col_from_description("FIELDS").alias("source_field_name"),
        clean_string_col_from_description("DESCRIPTION").alias("field_description")
    )
    .withColumn(
        "standardized_field_name",
        F.lower(
            F.regexp_replace(
                F.regexp_replace(F.col("source_field_name"), "[^A-Za-z0-9]+", "_"),
                "_+",
                "_"
            )
        )
    )
    .withColumn(
        "standardized_field_name",
        F.regexp_replace(F.col("standardized_field_name"), "^_|_$", "")
    )
    .withColumn(
        "contains_pii_signal",
        F.when(
            F.lower(F.col("source_field_name")).contains("email")
            | F.lower(F.col("source_field_name")).contains("fname")
            | F.lower(F.col("source_field_name")).contains("lname")
            | F.lower(F.col("source_field_name")).contains("password")
            | F.lower(F.col("source_field_name")).contains("street")
            | F.lower(F.col("source_field_name")).contains("zipcode")
            | F.lower(F.col("source_field_name")).contains("latitude")
            | F.lower(F.col("source_field_name")).contains("longitude"),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn("source_table", F.lit(DESCRIPTION_SOURCE_TABLE))
    .withColumn("dictionary_record_id", F.sha2(F.concat_ws("||", "source_field_name", "field_description"), 256))
    .withColumn("silver_transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("silver_created_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("silver_created_by", F.lit(CREATED_BY))
    .withColumn("silver_notebook_name", F.lit(NOTEBOOK_NAME))
)

silver_dataco_dictionary_row_count = silver_dataco_dictionary_df.count()

print("Silver DataCo field dictionary row count:", silver_dataco_dictionary_row_count)

display(
    silver_dataco_dictionary_df
    .select(
        "dictionary_record_id",
        "source_field_name",
        "standardized_field_name",
        "field_description",
        "contains_pii_signal"
    )
    .orderBy("source_field_name")
)

silver_dataco_dictionary_df.printSchema()

In [0]:
# Notebook 11, Chunk 10: Validate and write Silver DataCo field dictionary

dictionary_validation_rows = []

source_dictionary_row_count = bronze_description_df.count()
target_dictionary_row_count = silver_dataco_dictionary_df.count()

dictionary_validation_rows.append({
    "validation_check": "dictionary_row_count_preserved",
    "expected_value": str(source_dictionary_row_count),
    "actual_value": str(target_dictionary_row_count),
    "status": "PASS" if source_dictionary_row_count == target_dictionary_row_count else "FAIL",
    "issue_detail": None if source_dictionary_row_count == target_dictionary_row_count else "Silver dictionary row count does not match Bronze description row count."
})

null_field_name_count = (
    silver_dataco_dictionary_df
    .filter(F.col("source_field_name").isNull())
    .count()
)

dictionary_validation_rows.append({
    "validation_check": "source_field_name_not_null",
    "expected_value": "0 null source field names",
    "actual_value": str(null_field_name_count),
    "status": "PASS" if null_field_name_count == 0 else "FAIL",
    "issue_detail": None if null_field_name_count == 0 else "Null source_field_name values found."
})

null_description_count = (
    silver_dataco_dictionary_df
    .filter(F.col("field_description").isNull())
    .count()
)

dictionary_validation_rows.append({
    "validation_check": "field_description_not_null",
    "expected_value": "0 null field descriptions",
    "actual_value": str(null_description_count),
    "status": "PASS" if null_description_count == 0 else "FAIL",
    "issue_detail": None if null_description_count == 0 else "Null field_description values found."
})

duplicate_field_count = (
    silver_dataco_dictionary_df
    .groupBy("source_field_name")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

dictionary_validation_rows.append({
    "validation_check": "source_field_name_unique",
    "expected_value": "0 duplicate source field names",
    "actual_value": str(duplicate_field_count),
    "status": "PASS" if duplicate_field_count == 0 else "FAIL",
    "issue_detail": None if duplicate_field_count == 0 else "Duplicate source_field_name values found."
})

pii_signal_count = (
    silver_dataco_dictionary_df
    .filter(F.col("contains_pii_signal") == True)
    .count()
)

dictionary_validation_rows.append({
    "validation_check": "pii_signal_detection_present",
    "expected_value": "At least 1 PII-like field flagged",
    "actual_value": str(pii_signal_count),
    "status": "PASS" if pii_signal_count > 0 else "WARN",
    "issue_detail": None if pii_signal_count > 0 else "No PII-like fields were flagged in the dictionary."
})

dictionary_validation_schema = T.StructType([
    T.StructField("validation_check", T.StringType(), True),
    T.StructField("expected_value", T.StringType(), True),
    T.StructField("actual_value", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("issue_detail", T.StringType(), True)
])

dictionary_validation_df = spark.createDataFrame(
    dictionary_validation_rows,
    schema=dictionary_validation_schema
)

display(dictionary_validation_df.orderBy("status", "validation_check"))

dictionary_fail_count = dictionary_validation_df.filter(F.col("status") == "FAIL").count()

print("Silver DataCo dictionary validation failures:", dictionary_fail_count)

if dictionary_fail_count == 0:
    (
        silver_dataco_dictionary_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DESCRIPTION_TARGET_TABLE)
    )

    print(f"✅ Wrote Silver DataCo field dictionary table: {DESCRIPTION_TARGET_TABLE}")
else:
    raise ValueError("Silver DataCo dictionary validation failed. Fix issues before writing.")

In [0]:
written_dictionary_df = spark.table(DESCRIPTION_TARGET_TABLE)

written_dictionary_row_count = written_dictionary_df.count()

print("Written Silver DataCo dictionary row count:", written_dictionary_row_count)

display(
    written_dictionary_df
    .select(
        "source_field_name",
        "standardized_field_name",
        "field_description",
        "contains_pii_signal"
    )
    .orderBy("source_field_name")
)

if written_dictionary_row_count == 52:
    print("✅ Read-back check passed: Silver DataCo field dictionary exists and row count is correct.")
else:
    print("❌ Read-back check failed: Row count does not match expected 52.")

In [0]:
dictionary_validation_results_df = (
    dictionary_validation_df
    .withColumn("transform_run_id", F.lit(TRANSFORM_RUN_ID))
    .withColumn("source_table", F.lit(DESCRIPTION_SOURCE_TABLE))
    .withColumn("target_table", F.lit(DESCRIPTION_TARGET_TABLE))
    .withColumn("validated_at", F.lit(datetime.now(timezone.utc).isoformat()))
    .withColumn("created_by", F.lit(CREATED_BY))
    .withColumn("notebook_name", F.lit(NOTEBOOK_NAME))
    .select(
        "transform_run_id",
        "source_table",
        "target_table",
        "validation_check",
        "expected_value",
        "actual_value",
        "status",
        "issue_detail",
        "validated_at",
        "created_by",
        "notebook_name"
    )
)

validation_results_table = f"{SILVER_SCHEMA}.silver_transform_validation_results"

(
    dictionary_validation_results_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(validation_results_table)
)

display(
    spark.table(validation_results_table)
    .filter(F.col("transform_run_id") == TRANSFORM_RUN_ID)
    .groupBy("target_table", "status")
    .agg(F.count("*").alias("validation_check_count"))
    .orderBy("target_table", "status")
)

print("✅ Notebook 11 fully complete: DataCo orders/shipments and field dictionary created and validated.")
print("Next notebook: 12_silver_synthetic_internal_tables")